<a href="https://colab.research.google.com/github/Odreybas/Data-and-ai/blob/main/week_12_Assignment_Generative_AI_and_RAG_name_Eric_Mwangi_cyberShujaa_id_cs_da03_26045.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

week 12 Assignment:Generatoive AI and RAG
name:Eric Mwangi
cyberShujaa id:cs-da03-26045

In [1]:

!pip install -q langchain langchain-community langchain-huggingface transformers sentence-transformers faiss-cpu pypdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [3]:
#imports

import re
import torch
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM



In [4]:
#text  cleaning function

def fix_pdf_text(text):
    text = re.sub(r'(?<= )([^ ]) (?=[^ ])', r'\1', text)
    text = re.sub(r'^([^ ]) (?=[^ ])', r'\1', text)
    text = re.sub(r'\n([^ ]) ', r'\n\1', text)
    return re.sub(r'\s+', ' ', text).strip()


In [5]:
#upload and red pdf

print('Please upload your PDF file:')
uploaded = files.upload()

if uploaded:
    pdf_path = list(uploaded.keys())[0]

    loader = PyPDFLoader(pdf_path)
    pages = loader.load()

    for page in pages:
        page.page_content = fix_pdf_text(page.page_content)



Please upload your PDF file:


Saving Eric_Kinyua_Mwangi__CV.pdf to Eric_Kinyua_Mwangi__CV.pdf


In [7]:
#spliting into chunks

splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=100)
chunks = splitter.split_documents(pages)

print(f"Total chunks created: {len(chunks)}")

Total chunks created: 5


In [9]:
#embeddings and ve ctor strore
print('Creating embeddings and vector store...')

embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

vectorstore = FAISS.from_documents(chunks, embeddings)

retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

Creating embeddings and vector store...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
#loading the llm

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = 'google/flan-t5-large'

print(f'Loading {model_id} on {device}...')

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)

Loading google/flan-t5-large on cpu...


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [17]:
#rag guery function

def query_rag(question):
    relevant_docs = retriever.invoke(question)
    context = "\n".join([doc.page_content for doc in relevant_docs])

    input_text = f"answer: {question} context: {context}"

    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=1024).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [18]:
#run a query

def query_rag(question):
    relevant_docs = retriever.invoke(question)
    context = "\n".join([doc.page_content for doc in relevant_docs])

    input_text = f"answer: {question} context: {context}"

    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=1024).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

**testing**

In [19]:
question = "Summarize this document."
response = query_rag(question)
print(response)

The Kenyan government is looking for a graduate to join its Artificial Intelligence team.


In [20]:
question = "What are the key skills mentioned in the document?"
response = query_rag(question)
print(f"Question: {question}\nResponse: {response}\n")

Question: What are the key skills mentioned in the document?
Response: gaining industry exposure, continuously improving technical skills, and



In [21]:
question = "What is the candidate's work experience?"
response = query_rag(question)
print(f"Question: {question}\nResponse: {response}\n")

Question: What is the candidate's work experience?
Response: Online Shopping System (Web Project)



In [22]:
question = "What educational background does the candidate have?"
response = query_rag(question)
print(f"Question: {question}\nResponse: {response}\n")

Question: What educational background does the candidate have?
Response: growing within aprofessional environment



In [23]:
question = "What projects has the candidate worked on?"
response = query_rag(question)
print(f"Question: {question}\nResponse: {response}\n")

Question: What projects has the candidate worked on?
Response: Online Shopping System



In [24]:
question = "Is there any contact information provided in the document?"
response = query_rag(question)
print(f"Question: {question}\nResponse: {response}\n")

Question: Is there any contact information provided in the document?
Response: Email: dweshwesh1@gmail.com Phone:07011514752

